# RAG-Based Educational Chatbot
### Assignment: Building a Simple RAG-Based Educational Chatbot for School Students

**Subjects Covered:** Mathematics | Science | English | Social Studies

---

## Task 1 — RAG Pipeline Design

The complete RAG pipeline works as follows:

```
[Educational Documents]
        ↓
[Document Loading]       ← Load .txt / .pdf study materials
        ↓
[Text Chunking]          ← Split into manageable pieces
        ↓
[Embedding Generation]   ← Convert chunks to vectors
        ↓
[Vector Database Storage] ← Store vectors (ChromaDB)
        ↓
[Student Question] → [Similarity Search] → [Top-K Chunks]
        ↓
[Prompt Engineering]     ← Inject retrieved context
        ↓
[LLM (OpenAI GPT)] → Student-Friendly Answer
```

**Key design decisions:**
- Use `sentence-transformers` for free, local embeddings
- Use `ChromaDB` as a lightweight in-memory vector store
- Use `OpenAI GPT-3.5-turbo` as the LLM for answer generation
- Retrieve top-3 most relevant chunks per question

## Install Dependencies

In [ ]:
# Install all required libraries
!pip install chromadb sentence-transformers openai

## Task 2 — Chunking Strategy

**Chosen Strategy: Sliding Window (Fixed-size with Overlap)**

| Parameter | Value | Reason |
|-----------|-------|--------|
| Chunk size | 300 tokens (~1–2 paragraphs) | Fits one concept per chunk |
| Overlap | 50 tokens | Preserves context at boundaries |

**Why this suits textbooks:**
- Textbooks explain one concept per paragraph — 300 tokens captures that naturally
- Overlap ensures sentences cut mid-idea are still retrievable
- Too large → noisy retrieval; Too small → incomplete answers

**Impact on retrieval quality:** Good chunk size = precise semantic match, fewer irrelevant passages returned.

In [ ]:
# Task 2 — Text Chunking Function

def chunk_text(text, chunk_size=300, overlap=50):
    """
    Splits text into overlapping chunks.
    chunk_size: number of words per chunk
    overlap: number of words shared between consecutive chunks
    """
    words = text.split()
    chunks = []
    start = 0

    while start < len(words):
        end = start + chunk_size
        chunk = " ".join(words[start:end])
        chunks.append(chunk)
        start += chunk_size - overlap  # slide forward with overlap

    return chunks


# Quick test
sample = "Photosynthesis is the process by which plants use sunlight water and carbon dioxide to produce oxygen and energy in the form of glucose. " * 10
chunks = chunk_text(sample)
print(f"Total chunks created: {len(chunks)}")
print(f"\nSample chunk:\n{chunks[0]}")

## Task 3 — Embedding Model & Vector Database

**Embedding Model: `all-MiniLM-L6-v2` (sentence-transformers)**
- Lightweight, fast, free — runs locally without an API key
- Trained on diverse text including educational content
- Produces 384-dimensional dense vectors — good balance of speed vs quality
- Embeddings capture *semantic meaning*, so "H₂O" and "water molecule" map to nearby vectors

**Vector Database: ChromaDB**
- Zero setup — runs fully in-memory (no server needed)
- Built-in similarity search using cosine distance
- Perfect for a school-scale dataset (thousands of chunks)
- Easy Python API, well-suited for notebooks

In [ ]:
# Task 3 — Load Embedding Model and Set Up Vector Database

import chromadb
from sentence_transformers import SentenceTransformer

# Load embedding model
print("Loading embedding model...")
embedding_model = SentenceTransformer("all-MiniLM-L6-v2")
print("Embedding model loaded!")

# Create in-memory ChromaDB
chroma_client = chromadb.Client()
collection = chroma_client.create_collection(
    name="educational_docs",
    metadata={"hnsw:space": "cosine"}  # cosine similarity
)

print("ChromaDB collection ready!")

## Task 4 — Prompt Engineering

**Prompt Template Design:**

```
You are a helpful tutor for school students...
Context: {retrieved_chunks}
Question: {student_question}
Answer: ...
```

**Design Reasoning:**
| Element | Purpose |
|---|---|
| Role: "helpful tutor" | Sets friendly, educational tone |
| "Only use the context below" | Prevents hallucination |
| "If unsure, say so" | Honest fallback to avoid making up facts |
| "Simple language" | Makes answers accessible to school students |
| "Step by step if needed" | Helps with math/science explanations |

In [ ]:
# Task 4 — Prompt Template

def build_prompt(context_chunks, student_question):
    """
    Builds a prompt using retrieved context and the student's question.
    """
    context = "\n\n".join(context_chunks)

    prompt = f"""You are a helpful and friendly tutor for school students.
Your job is to answer the student's question using ONLY the context provided below.
- Use simple, easy-to-understand language suitable for school students.
- Explain step by step if the question involves a process or calculation.
- If the answer is not in the context, say: "I'm not sure about that — please check your textbook."
- Do NOT make up information.

--- CONTEXT ---
{context}
---------------

Student's Question: {student_question}

Answer:"""

    return prompt


# Preview the prompt structure
sample_prompt = build_prompt(
    context_chunks=["Photosynthesis occurs in the chloroplast of plant cells..."],
    student_question="What is photosynthesis?"
)
print(sample_prompt)

## Task 5 — Build the Chatbot

### Step 5a: Load Educational Documents

In [ ]:
# Step 5a — Load Educational Documents
# For this demo, we use sample study material as strings.
# In a real setup, replace this with: open('textbook.txt').read() or use PyMuPDF for PDFs.

educational_documents = {
    "science": """
    Photosynthesis is the process by which green plants and some other organisms use sunlight
    to synthesize nutrients from carbon dioxide and water. Photosynthesis in plants generally
    involves the green pigment chlorophyll and generates oxygen as a byproduct.
    The equation for photosynthesis is: 6CO2 + 6H2O + light energy → C6H12O6 + 6O2.
    This process takes place mainly in the leaves of the plant inside the chloroplasts.
    Chlorophyll absorbs sunlight primarily in the red and blue wavelengths.

    The water cycle, also known as the hydrological cycle, describes the continuous movement
    of water on, above and below the surface of the Earth. Water evaporates from oceans and
    lakes, rises as water vapour, condenses into clouds, and falls back as precipitation.
    """,

    "mathematics": """
    A fraction represents a part of a whole. It has two parts: the numerator (top number)
    and denominator (bottom number). For example, 3/4 means 3 parts out of 4 equal parts.
    To add fractions with the same denominator, simply add the numerators: 1/4 + 2/4 = 3/4.
    To add fractions with different denominators, first find the LCM (Least Common Multiple)
    of the denominators, convert to equivalent fractions, then add.

    The Pythagorean theorem states that in a right-angled triangle, the square of the
    hypotenuse equals the sum of squares of the other two sides: a² + b² = c².
    This is only valid for right-angled triangles. Example: if a=3 and b=4, then c=5.
    """,

    "english": """
    A noun is a word that names a person, place, thing, or idea. Examples include:
    teacher, school, book, happiness. Nouns can be singular (one item) or plural (more than one).
    Common nouns refer to general things (city, dog), while proper nouns name specific things (London, Max).

    An adjective is a word that describes or modifies a noun. It tells us more about the noun
    such as its size, color, shape, or quality. Examples: big, red, happy, tall.
    Adjectives usually come before the noun they describe: 'a beautiful flower'.
    """,

    "social_studies": """
    Democracy is a system of government in which power is vested in the people.
    Citizens elect their representatives through free and fair elections.
    There are two main types: direct democracy (citizens vote on laws directly)
    and representative democracy (elected officials make decisions on behalf of citizens).
    India is the world's largest democracy. Elections are held every 5 years.

    The French Revolution began in 1789 and ended the monarchy in France.
    It was caused by financial crisis, social inequality, and the influence of Enlightenment ideas.
    The revolution introduced the ideals of Liberty, Equality, and Fraternity.
    """
}

print("Documents loaded for subjects:", list(educational_documents.keys()))

### Step 5b: Chunk, Embed, and Store in Vector DB

In [ ]:
# Step 5b — Chunk documents, generate embeddings, store in ChromaDB

all_chunks = []
all_ids = []
all_metadata = []

for subject, text in educational_documents.items():
    chunks = chunk_text(text, chunk_size=60, overlap=10)  # smaller size for our short demo texts
    for i, chunk in enumerate(chunks):
        all_chunks.append(chunk)
        all_ids.append(f"{subject}_chunk_{i}")
        all_metadata.append({"subject": subject})

print(f"Total chunks: {len(all_chunks)}")

# Generate embeddings
print("Generating embeddings...")
embeddings = embedding_model.encode(all_chunks).tolist()
print(f"Embeddings shape: {len(embeddings)} vectors of size {len(embeddings[0])}")

# Store in ChromaDB
collection.add(
    documents=all_chunks,
    embeddings=embeddings,
    ids=all_ids,
    metadatas=all_metadata
)

print("All chunks stored in vector database!")

### Step 5c: Retrieval Function

In [ ]:
# Step 5c — Retrieve relevant chunks for a student question

def retrieve_relevant_chunks(question, top_k=3):
    """
    Embeds the question and finds the top_k most similar chunks.
    """
    question_embedding = embedding_model.encode([question]).tolist()

    results = collection.query(
        query_embeddings=question_embedding,
        n_results=top_k
    )

    chunks = results["documents"][0]  # list of top-k chunk texts
    return chunks


# Test retrieval
test_question = "What is photosynthesis?"
retrieved = retrieve_relevant_chunks(test_question)
print(f"Top chunks for: '{test_question}'\n")
for i, chunk in enumerate(retrieved, 1):
    print(f"[Chunk {i}]: {chunk}\n")

### Step 5d: Set OpenAI API Key

In [ ]:
# Step 5d — Set your OpenAI API Key
import openai

openai.api_key = "sk-..."  # ← Replace with your actual OpenAI API key

print("OpenAI client ready!")

### Step 5e: LLM Answer Generation (OpenAI GPT)

In [ ]:
# Step 5e — Generate answer using OpenAI GPT-3.5-turbo

def generate_answer(question):
    """
    Full RAG pipeline:
    1. Retrieve relevant chunks
    2. Build prompt with context
    3. Send to OpenAI GPT and return answer
    """
    # Step 1: Retrieve
    context_chunks = retrieve_relevant_chunks(question, top_k=3)

    # Step 2: Build prompt
    prompt = build_prompt(context_chunks, question)

    # Step 3: Call OpenAI
    response = openai.chat.completions.create(
        model="gpt-3.5-turbo",
        messages=[
            {"role": "user", "content": prompt}
        ],
        max_tokens=512,
        temperature=0.3  # lower = more factual, less creative
    )

    return response.choices[0].message.content


# Test it!
answer = generate_answer("What is photosynthesis?")
print(answer)

### Step 5f: Interactive Chatbot Loop

In [ ]:
# Step 5f — Simple interactive chatbot (type 'quit' to exit)

print("="*55)
print("  Welcome to the Educational Chatbot!")
print("  Ask questions on Science, Math, English, Social Studies")
print("  Type 'quit' to exit")
print("="*55)

while True:
    question = input("\nYou: ").strip()

    if question.lower() in ["quit", "exit", "q"]:
        print("Goodbye! Happy studying!")
        break

    if not question:
        continue

    print("\nChatbot: Thinking...")
    answer = generate_answer(question)
    print(f"\nChatbot: {answer}")
    print("-" * 55)

---
## Sample Questions to Try

```
Science:        What is the water cycle?
Mathematics:    How do I add fractions with different denominators?
Mathematics:    What is the Pythagorean theorem?
English:        What is an adjective?
Social Studies: What is democracy?
Social Studies: What caused the French Revolution?
```